# Funding model retraining study

v2 froze one retraining policy, refit once per fold on all prior data, and found the model adds nothing over persistence in the strategy. That policy was never varied. This asks whether it matters: does the training window (expanding all history versus a rolling fixed length) or the refit cadence (how often the model is retrained) move the downstream PnL, and can a better-retrained model finally beat persistence.

Signal-side only, no engine work. The judge is downstream net PnL and Sharpe of the two shapes that won in v2, fixed threshold and slow partial adjustment, run on the same window and cost. Not IC. v2 showed IC does not translate to money. Persistence is invariant to the policy and is the reference line on every run.

## Setup

The engine module, the ten symbols, the cost table. Same feed and cost as v2.

In [1]:
import os, sys, glob, warnings, time
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
from scipy.stats import spearmanr
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.pipeline import Pipeline

REPO = Path.cwd()
while REPO != REPO.parent and not (REPO / "research").is_dir():
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
import research.funding_carry.features as F
import research.cv as CV
from research.funding_carry.models import _with_symbol_dummies

_so = glob.glob(str(REPO / "build" / "release" / "**" / "qp_python_backtest*.so"), recursive=True)
if not _so:
    raise RuntimeError("qp_python_backtest*.so not found; build the target first")
sys.path.insert(0, os.path.dirname(_so[0]))
import qp_python_backtest as qb

DATA       = REPO / "data" / "binance_historical"
COST_TABLE = str(REPO / "data" / "cost_model" / "cost_table.csv")
SYMBOLS = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "BNBUSDT", "XRPUSDT",
           "DOGEUSDT", "ADAUSDT", "LINKUSDT", "AVAXUSDT", "LTCUSDT"]
NAME_TO_ID = {n: i for i, n in enumerate(SYMBOLS)}
ID_TO_NAME = {i: n for n, i in NAME_TO_ID.items()}
FUT, SPOT = 0, 1
HORIZON = 24

## Feature frame

Same causal feature chain as the frozen v2 predictor, funding and premium events, lags and ewmas, kline aggregates, basket and interaction features. Target is realized cumulative funding over the next 24 8h prints. 32058 rows, 10 symbols, 2022 to 2024.

In [2]:
FUNDING_COLS = ["symbol", "tag", "ts_ms", "interval_h", "funding_rate"]
PREMIUM_COLS = ["ts_ms", "open", "high", "low", "close", "volume", "close_ms",
                "quote_vol", "trades", "taker_buy_base", "taker_buy_quote", "ignore"]


def load_funding(sym):
    d = DATA / sym / "futures" / "funding"
    files = sorted(d.glob(f"{sym}-fundingRate-*.csv"))
    df = pd.concat([pd.read_csv(f, header=None, names=FUNDING_COLS) for f in files],
                   ignore_index=True)
    df["ts"] = pd.to_datetime(df["ts_ms"], unit="ms", utc=True)
    df["symbol"] = sym
    df = df[["symbol", "ts", "funding_rate"]].rename(columns={"funding_rate": "realized_funding"})
    return df.sort_values("ts").drop_duplicates("ts").reset_index(drop=True)


def load_premium(sym):
    d = DATA / sym / "futures" / "premiumindex"
    files = sorted(d.glob("*.csv"))
    if not files:
        return pd.DataFrame(columns=["ts_ms", "close"])
    parts = [pd.read_csv(f, header=None, names=PREMIUM_COLS)[["ts_ms", "close"]] for f in files]
    df = pd.concat(parts, ignore_index=True).sort_values("ts_ms").drop_duplicates("ts_ms").reset_index(drop=True)
    df["ts"] = pd.to_datetime(df["ts_ms"], unit="ms", utc=True)
    return df


def average_premium_up_to(premium, funding_ts, interval_hours=8):
    if premium.empty:
        return pd.Series(np.nan, index=range(len(funding_ts)))
    p = premium.sort_values("ts").reset_index(drop=True)
    win = pd.Timedelta(hours=interval_hours)
    out = np.full(len(funding_ts), np.nan)
    ts_vals = pd.to_datetime(funding_ts, utc=True).to_numpy()
    p_ts = p["ts"].to_numpy(); p_close = p["close"].to_numpy(dtype=float)
    for i, end in enumerate(ts_vals):
        beg = end - win
        lo = np.searchsorted(p_ts, beg, side="right")
        hi = np.searchsorted(p_ts, end, side="right")
        if hi > lo:
            out[i] = p_close[lo:hi].mean()
    return pd.Series(out)


parts = []
for sym in SYMBOLS:
    f = load_funding(sym); p = load_premium(sym)
    f["premium"] = average_premium_up_to(p, f["ts"]).to_numpy()
    parts.append(f)
events = pd.concat(parts, ignore_index=True)

feat = events.copy()
for k in (1, 2, 3):
    feat = F.add_funding_lag(feat, k)
feat = F.add_funding_ewma(feat, halflife=2); feat = F.add_funding_ewma(feat, halflife=6)
feat = F.add_funding_vol(feat, window=8); feat = F.add_clamp_distance(feat)
feat = F.add_premium_trend(feat, span=3); feat = F.add_cross_symbol_spread(feat, reference="BTCUSDT")
feat = F.add_basket_spread(feat); feat = F.add_time_features(feat)
feat = F.add_funding_mean_window(feat, window=90); feat = F.add_funding_sign_window(feat, window=90)
feat = F.add_funding_vol_rank(feat, vol_window=30, rank_window=180)
kline_agg = pd.read_pickle(REPO / "research" / "funding_carry" / "results" / "kline_aggregates_8h.pkl")
feat = feat.merge(kline_agg, on=["symbol", "ts"], how="left")
feat["taker_imb_x_ewma2"] = feat["taker_imbalance"] * feat["funding_ewma_h2"]
feat["taker_imb_x_lag1"] = feat["taker_imbalance"] * feat["funding_lag1"]
feat["vol1m_x_clamp"] = feat["realized_vol_1m"] * feat["clamp_distance"]
feat["range_x_ewma2"] = feat["high_low_range"] * feat["funding_ewma_h2"]
feat["regime_x_lag1"] = feat["funding_sign_w90"] * feat["funding_lag1"]
feat["basket_x_lag1"] = feat["basket_spread"] * feat["funding_lag1"]
feat["lag1_sq"] = feat["funding_lag1"] ** 2
feat = F.add_basket_zscore(feat, source="funding_lag1")
feat = F.add_basket_zscore(feat, source="funding_ewma_h6")
feat = F.add_basket_rank(feat, source="funding_lag1")
feat["basket_z_x_lag1"] = feat["basket_z_funding_lag1"] * feat["funding_lag1"]
feat["basket_rank_x_lag1"] = feat["basket_rank_funding_lag1"] * feat["funding_lag1"]
feat = F.add_cum_target(feat, horizon=HORIZON)
FCOLS = [c for c in feat.columns
         if c not in ("symbol", "ts", "realized_funding", "premium", "realized_cum")]
feat = feat.dropna(subset=FCOLS + ["realized_funding", "realized_cum"]).reset_index(drop=True)
feat["year"] = feat["ts"].dt.year

# Same five walk-forward folds as v2. Fold id labels each test row; -1 is the
# pre-test warmup that never gets a prediction. Policy is chosen on folds 0 to
# 2 and reported on 3 and 4, so the choice never sees the reporting years.
folds = list(CV.walk_forward_splits(feat, n_folds=5, horizon=HORIZON, embargo=5))
fold_of = np.full(len(feat), -1)
for fi, (_, te) in enumerate(folds):
    fold_of[te] = fi
feat["fold"] = fold_of
print("feat", feat.shape, "features", len(FCOLS))
for fi in range(5):
    g = feat[feat.fold == fi]
    print(f"fold {fi}: {g.ts.min().date()} to {g.ts.max().date()}, {len(g)} rows")

feat (32058, 38) features 31
fold 0: 2022-12-07 to 2023-05-05, 4490 rows
fold 1: 2023-05-05 to 2023-10-02, 4490 rows
fold 2: 2023-10-02 to 2024-02-28, 4490 rows
fold 3: 2024-02-29 to 2024-07-27, 4488 rows
fold 4: 2024-07-27 to 2024-12-24, 4490 rows


## Retraining policy

One causal generator produces an out-of-fold prediction for every test row, walking forward through refit dates. At each refit date the model is trained only on the past, then predicts every row until the next refit. The policy is two knobs.

`window_months` is the training window. `None` is expanding, all history before the refit date. A number is a rolling window of that many months.

`refit_freq` is the cadence, a pandas offset. `MS` refits on the first of each month, `QS` each quarter.

Training rows within the 24-print label horizon of the refit date are purged, their label window reaches past the date into the prediction region. Persistence carries no model so it is the same under every policy, generated once as 24 times the last print.

In [3]:
PURGE = pd.Timedelta(hours=HORIZON * 8)   # 24-print label horizon


def elastic_pred(train, test, cols):
    Xtr, _ = _with_symbol_dummies(train, cols)
    pipe = Pipeline([("sc", StandardScaler()),
                     ("m", ElasticNet(alpha=1e-4, l1_ratio=0.7, max_iter=20000))])
    pipe.fit(Xtr, train["realized_cum"].to_numpy())
    Xte, _ = _with_symbol_dummies(test, cols)
    return pipe.predict(Xte)


def gbm_pred(train, test, cols):
    import lightgbm as lgb
    Xtr, c = _with_symbol_dummies(train, cols)
    y = train["realized_cum"].to_numpy()
    sp = int(len(y) * 0.8)
    p = dict(objective="regression", verbose=-1, feature_fraction=0.9,
             bagging_fraction=0.9, bagging_freq=5, learning_rate=0.02,
             num_leaves=15, min_data_in_leaf=500)
    dtr = lgb.Dataset(Xtr[:sp], label=y[:sp], feature_name=c)
    dval = lgb.Dataset(Xtr[sp:], label=y[sp:], reference=dtr)
    b = lgb.train(p, dtr, num_boost_round=3000, valid_sets=[dval],
                  callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])
    Xte, _ = _with_symbol_dummies(test, cols)
    return b.predict(Xte, num_iteration=b.best_iteration)


MODELS = {"elastic": elastic_pred, "gbm": gbm_pred}


def gen_policy_oof(window_months, refit_freq, min_train=2000):
    """Causal out-of-fold predictions for one retraining policy. Refit on the
    cadence, train on the past (rolling or expanding), predict to the next
    refit. Returned in memory, never persisted per config."""
    test = feat[feat.fold >= 0]
    start = pd.Timestamp(test.ts.min()).tz_convert("UTC").normalize().replace(day=1)
    dates = pd.date_range(start, test.ts.max(), freq=refit_freq, tz="UTC")
    di = np.searchsorted(dates.to_numpy(), test.ts.to_numpy(), side="right") - 1
    out = test[["symbol", "ts", "fold", "year", "realized_funding",
                "realized_cum", "funding_lag1"]].copy()
    for name in MODELS:
        out["pred_" + name] = np.nan
    out["pred_persistence"] = (HORIZON * test["funding_lag1"]).to_numpy()
    n_refit = 0
    for k, d in enumerate(dates):
        sl = test[di == k]
        if sl.empty:
            continue
        tr = feat[feat.ts <= d - PURGE]
        if window_months is not None:
            tr = tr[tr.ts >= d - pd.DateOffset(months=window_months)]
        if len(tr) < min_train:
            continue
        for name, fn in MODELS.items():
            out.loc[sl.index, "pred_" + name] = fn(tr, sl, FCOLS)
        n_refit += 1
    return out, n_refit

## Strategies and the run

The two v2 winners. `PredThreshold` opens a delta-neutral leg when the predicted cumulative funding clears the 34 bp round-trip cost and flattens when it falls back to zero, the lowest-turnover rule. `PredPartialAdjust` sets an aim proportional to the prediction over the hurdle and drifts a slow 0.15 of the way each print, the v2 sweep's best fraction. Same `SimpleRiskGate`, same cost table.

`evaluate` takes one policy's predictions, runs both strategies on each of the three sources, and returns one summary row per (strategy, source): PnL, Sharpe, fee drag, and the 2023 and 2024 splits. Nothing is persisted per config.

In [4]:
class PredThreshold:
    """Open at the 34 bp hurdle, flatten at zero, hold between. One lookup per
    funding print."""

    def __init__(self, bt, pred, qty=1.0, enter=0.0034, exit=0.0):
        self.bt = bt; self.pred = pred; self.qty = qty
        self.enter = enter; self.exit = exit

    def on_event(self, ev):
        if ev.kind != qb.EventKind.Funding or ev.venue != FUT:
            return None
        p = self.pred.get((ev.symbol, ev.ts))
        if p is None:
            return None
        held = self.bt.position(ev.symbol, SPOT)
        target = self.qty if p >= self.enter else (0.0 if p <= self.exit else held)
        return [qb.Intent(ev.symbol, SPOT, target), qb.Intent(ev.symbol, FUT, -target)]

    def on_timer(self, now):
        return None


class PredPartialAdjust:
    """Aim proportional to the prediction over the hurdle, capped, drifting a
    slow fraction each print."""

    def __init__(self, bt, pred, hurdle=0.0034, cap=2.0, adjust=0.15):
        self.bt = bt; self.pred = pred; self.hurdle = hurdle
        self.cap = cap; self.adjust = adjust

    def on_event(self, ev):
        if ev.kind != qb.EventKind.Funding or ev.venue != FUT:
            return None
        p = self.pred.get((ev.symbol, ev.ts))
        if p is None:
            return None
        aim = float(np.clip(p / self.hurdle, 0.0, self.cap))
        cur = self.bt.position(ev.symbol, SPOT)
        target = cur + self.adjust * (aim - cur)
        return [qb.Intent(ev.symbol, SPOT, target), qb.Intent(ev.symbol, FUT, -target)]

    def on_timer(self, now):
        return None


class SimpleRiskGate:
    """Per-(symbol, venue) exposure cap. Drawdown kill left unarmed here."""

    def __init__(self, bt, tracked, max_position_qty=10.0):
        self.bt = bt; self.tracked = tracked
        self.max_position_qty = max_position_qty; self._id = 1

    def _next(self):
        i = self._id; self._id += 1; return i

    def check(self, intent):
        current = self.bt.position(intent.symbol, intent.venue)
        target = float(np.clip(intent.target_position,
                               -self.max_position_qty, self.max_position_qty))
        delta = target - current
        outcome = (qb.RiskOutcome.Approved if target == intent.target_position
                   else qb.RiskOutcome.Resized)
        side = qb.Side.Buy if delta >= 0 else qb.Side.Sell
        return qb.RiskDecision(outcome, qb.Order(self._next(), intent.symbol,
                                                 side, intent.venue, abs(delta)))


PRED_FIRST = feat[feat.fold >= 0].ts.min().date().isoformat()
PRED_LAST  = feat[feat.fold >= 0].ts.max().date().isoformat()
SOURCES = ["persistence", "elastic", "gbm"]
STRATS = {"threshold": lambda bt, pm: PredThreshold(bt, pm, 1.0, 0.0034, 0.0),
          "partial":   lambda bt, pm: PredPartialAdjust(bt, pm, 0.0034, 2.0, 0.15)}


def pred_map(oof, col):
    d = oof.dropna(subset=[col])
    sid = d["symbol"].map(NAME_TO_ID).to_numpy()
    tns = (d["ts"].dt.tz_convert("UTC").dt.tz_localize(None)
           .values.astype("datetime64[ns]").astype("int64"))
    return dict(zip(zip(sid, tns), d[col]))


def run_backtest(pm, make_strategy):
    ds = qb.Dataset(str(DATA), SYMBOLS, PRED_FIRST, PRED_LAST, COST_TABLE)
    bt = qb.PythonBacktest(ds)
    strat = make_strategy(bt, pm)
    tracked = [(NAME_TO_ID[s], v) for s in SYMBOLS for v in (FUT, SPOT)]
    gate = SimpleRiskGate(bt, tracked, 10.0)
    bt.set_on_event(strat.on_event); bt.set_on_timer(strat.on_timer)
    bt.set_check(gate.check); bt.set_event_kinds([qb.EventKind.Funding])
    res = bt.run()
    eq = pd.DataFrame({"ts": pd.to_datetime([p.ts for p in res.equity_series], utc=True),
                       "equity": [p.equity for p in res.equity_series]})
    fees = float(np.sum(res.fees))
    return eq, fees, res.final_equity


def daily_sharpe(eq):
    d = eq.set_index("ts")["equity"].resample("1D").last().ffill().diff().dropna()
    return (d.mean() / d.std()) * np.sqrt(365) if d.std() else 0.0


def year_pnl(eq, year):
    e = eq.set_index("ts")["equity"]; seg = e[e.index.year == year]
    return float(seg.iloc[-1] - seg.iloc[0]) if len(seg) > 1 else 0.0


def evaluate(cfg_id, oof):
    rows = []
    for col in SOURCES:
        pm = pred_map(oof, "pred_" + col)
        for sname, mk in STRATS.items():
            eq, fees, fe = run_backtest(pm, mk)
            rows.append({"config": cfg_id, "strategy": sname, "source": col,
                         "pnl": round(fe, 1), "sharpe": round(daily_sharpe(eq), 2),
                         "fees": round(fees, 1),
                         "pnl_2023": round(year_pnl(eq, 2023), 1),
                         "pnl_2024": round(year_pnl(eq, 2024), 1)})
    return pd.DataFrame(rows)

## Coarse contrast, does the policy move PnL at all

Three configs against the binary question. Expanding versus rolling 18 months at a monthly refit isolates the window. Expanding monthly versus expanding quarterly isolates the cadence. If both deltas sit inside noise the policy does not matter, and the answer is to use expanding for simplicity while persistence still stands.

In [5]:
COARSE = {"expand_M": (None, "MS"), "roll18_M": (18, "MS"), "expand_Q": (None, "QS")}

rows = []
for cid, (wm, rf) in COARSE.items():
    t0 = time.time()
    oof, nr = gen_policy_oof(wm, rf)
    rows.append(evaluate(cid, oof))
    print(f"[{cid}] refits={nr}  {time.time() - t0:.1f}s")
coarse = pd.concat(rows, ignore_index=True)

for sname in STRATS:
    sub = coarse[coarse.strategy == sname]
    print(f"\n=== {sname} ===")
    display(sub.pivot(index="source", columns="config",
                      values=["pnl", "sharpe", "fees"]).round(2))

[expand_M] refits=25  201.0s
[roll18_M] refits=25  218.3s
[expand_Q] refits=8  176.8s

=== threshold ===


pnl                     sharpe                       fees  \
config      expand_M expand_Q roll18_M expand_M expand_Q roll18_M expand_M   
source                                                                       
elastic       8692.2   8259.1   8291.9     4.65     4.47     4.64    271.0   
gbm           7843.3   7845.7   8694.3     4.17     4.17     4.31    155.2   
persistence   8580.0   8580.0   8580.0     4.97     4.97     4.97    380.1   

                               
config      expand_Q roll18_M  
source                         
elastic        192.6    447.3  
gbm            155.3     38.2  
persistence    380.1    380.1


=== partial ===


pnl                     sharpe                       fees  \
config      expand_M expand_Q roll18_M expand_M expand_Q roll18_M expand_M   
source                                                                       
elastic       9756.5   9270.3  10415.8     4.15     4.18     4.15   1425.3   
gbm          10437.8   9628.8  10134.4     4.41     4.54     4.39   1101.0   
persistence   9679.3   9679.3   9679.3     3.82     3.82     3.82   2426.8   

                               
config      expand_Q roll18_M  
source                         
elastic       1400.8   1466.3  
gbm           1088.8   1260.2  
persistence   2426.8   2426.8

The same grid split by year. A policy that is real is good in both years, a lone winner in one is a fit to that year.

In [6]:
for sname in STRATS:
    sub = coarse[coarse.strategy == sname]
    print(f"\n=== {sname}: pnl by year ===")
    display(sub.pivot(index="source", columns="config",
                      values=["pnl_2023", "pnl_2024"]).round(1))


=== threshold: pnl by year ===


pnl_2023                   pnl_2024                  
config      expand_M expand_Q roll18_M expand_M expand_Q roll18_M
source                                                           
elastic        725.3    294.9    556.9   7945.7   7942.9   7713.8
gbm              0.0      0.0      0.4   7882.2   7884.6   8732.7
persistence   1308.7   1308.7   1308.7   7250.0   7250.0   7250.0


=== partial: pnl by year ===


pnl_2023                   pnl_2024                  
config      expand_M expand_Q roll18_M expand_M expand_Q roll18_M
source                                                           
elastic       1148.6   1005.0   1192.8   8557.9   8252.6   9172.5
gbm           1231.5   1068.2   1208.2   9186.2   8557.5   8905.5
persistence   1326.9   1326.9   1326.9   8308.9   8308.9   8308.9

Persistence is identical down every column, the check that it carries no model and is the same line under every policy.

The policy barely moves anything. Window, expanding versus rolling 18 months at a monthly refit: elastic partial goes 9756 to 10416, gbm partial 10438 to 10134, and the threshold rule moves the other way, elastic 8692 to 8292, gbm 7843 to 8694. Rolling helps one cell and hurts the next, no consistent sign. Cadence, monthly versus quarterly on the expanding window, is smaller still, a few hundred either way with Sharpe flat.

Every move sits inside the gap between the two strategies and well inside the gap between sources. The spread across the whole grid for a fixed source and rule is 400 to 1200 on an 8000 to 10000 pnl, and the best config flips by cell, expanding for gbm partial, rolling for elastic partial and gbm threshold. A best window its neighbours do not agree with is noise. By year it is the same, no config leads both 2023 and 2024.

And no verdict changed. On the low-turnover threshold rule persistence at 8580 and Sharpe 5.0 is still the one to beat, no model config clears it by more than the noise. On partial adjust gbm edges persistence on pnl, 10438 to 9679, but it did that in v2 already, and still pays for it in Sharpe, 4.4 against 3.8. Retraining smarter did not make the model beat persistence where it did not before.

## Read

Retraining policy does not matter here. Neither the training window, pushed to a full 12 to 24 month and expanding ladder, nor the refit cadence moves downstream pnl or Sharpe past the noise between strategies, and no policy makes the model beat persistence on the rule where it does not already. The surface is flat, which is itself the answer.

Persistence stands. The live retrain schedule is a simplicity choice, not an edge: expanding window, refit monthly. No per-config prediction file is kept, the expanding causal set the strategy work already runs on is the one.

The reason is upstream of the policy. Funding is set mechanically and mean-reverts slowly, so it sits near its own last value and persistence already captures almost all of it. Retraining the model better cannot manufacture an edge the signal does not hold. The levers that move carry are cost, structure and risk, not prediction: maker execution on the spot leg, the short-funding side, and modelling the tail the sim omits. A one-time capture decomposition on the final model, funding available against funding harvested with fee drag, would localize exactly where the model misses, but it will confirm the thin timing lift, not overturn it.

## Fine, window length

The coarse contrast was flat, but flat is the load-bearing claim, so push the one axis with any pull, the training window, to a full ladder. 12, 18, 24 months and expanding, all at the monthly refit. 18 and expanding are already run above, so only 12 and 24 are new. If a real length exists its neighbours agree with it. If the best length jumps around by source and rule it is noise.

In [7]:
FINE = {"roll12_M": (12, "MS"), "roll24_M": (24, "MS")}

rows = []
for cid, (wm, rf) in FINE.items():
    t0 = time.time()
    oof, nr = gen_policy_oof(wm, rf)
    rows.append(evaluate(cid, oof))
    print(f"[{cid}] refits={nr}  {time.time() - t0:.1f}s")
fine = pd.concat(rows, ignore_index=True)

ORDER = ["roll12_M", "roll18_M", "roll24_M", "expand_M"]
ladder = pd.concat([coarse[coarse.config.isin(["roll18_M", "expand_M"])], fine],
                   ignore_index=True)
ladder["config"] = pd.Categorical(ladder["config"], ORDER)
for sname in STRATS:
    sub = ladder[ladder.strategy == sname]
    print(f"\n=== {sname} ===")
    display(sub.pivot_table(index="source", columns="config",
                            values=["pnl", "sharpe", "fees"], observed=True).round(2))

[roll12_M] refits=25  209.4s
[roll24_M] refits=25  197.1s

=== threshold ===


fees                                 pnl                    \
config      roll12_M roll18_M roll24_M expand_M roll12_M roll18_M roll24_M   
source                                                                       
elastic        344.9    447.3    268.1    271.0   9464.9   8291.9   8805.4   
gbm            155.8     38.2    156.4    155.2   8626.5   8694.3   7841.6   
persistence    380.1    380.1    380.1    380.1   8580.0   8580.0   8580.0   

                       sharpe                             
config      expand_M roll12_M roll18_M roll24_M expand_M  
source                                                    
elastic       8692.2     4.90     4.64     4.68     4.65  
gbm           7843.3     4.56     4.31     4.17     4.17  
persistence   8580.0     4.97     4.97     4.97     4.97


=== partial ===


fees                                 pnl                    \
config      roll12_M roll18_M roll24_M expand_M roll12_M roll18_M roll24_M   
source                                                                       
elastic       1473.2   1466.3   1481.5   1425.3  10152.2  10415.8   9892.8   
gbm           1362.7   1260.2   1149.4   1101.0   9921.1  10134.4  10346.2   
persistence   2426.8   2426.8   2426.8   2426.8   9679.3   9679.3   9679.3   

                       sharpe                             
config      expand_M roll12_M roll18_M roll24_M expand_M  
source                                                    
elastic       9756.5     4.38     4.15     4.14     4.15  
gbm          10437.8     4.43     4.39     4.41     4.41  
persistence   9679.3     3.82     3.82     3.82     3.82

The ladder is jagged. The best window is a different length in every cell, 12 for elastic threshold, 18 for gbm threshold and elastic partial, expanding for gbm partial, and the spread within a source and rule is 500 to 1200 on a 9 to 10k pnl, the same order as the coarse deltas. No length wins, and a best window its neighbours and its siblings do not agree on is noise.

The one coherent thread is gbm partial rising with window length, 9921 at 12 months to 10438 expanding, gbm uses all the history it is given. That points at expanding, the simplicity choice, not at a tuned window. Persistence is unchanged, still owning the threshold rule at 8580 and Sharpe 5.0.

Finer did not change it. The window is a non-lever, expanding monthly stands.